# Tổng quan và kiểm tra chất lượng dữ liệu PSPINES

Notebook này chỉ xuất số liệu tổng hợp, không in patient ID hay report text. Đường dẫn mặc định là repo local.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..') if Path('../dataset/dataset_master.csv').exists() else Path('.')
MASTER = ROOT / 'dataset/dataset_master.csv'
PATIENTS = ROOT / 'dataset/dataset_patients.jsonl'
master = pd.read_csv(MASTER)
patients = [json.loads(line) for line in PATIENTS.read_text(encoding='utf-8').splitlines() if line.strip()]
print({'rows': len(master), 'columns': len(master.columns), 'patients': master.patient_id.nunique(), 'json_records': len(patients)})

{'rows': 1235, 'columns': 42, 'patients': 247, 'json_records': 247}


In [2]:
fields = ['pfirrmann_grade','modic','disc_herniation','disc_bulging','disc_narrowing','spondylolisthesis','up_endplate','low_endplate']
profile = []
for field in fields:
    counts = master[field].value_counts(dropna=False).to_dict()
    profile.append({'field': field, 'distribution': {str(k): int(v) for k, v in counts.items()}, 'missing': int(master[field].isna().sum())})
pd.DataFrame(profile)

,field,distribution,missing
0,pfirrmann_grade,"{'2': 689, '3': 418, '4': 88, '1': 36, '5': 4}",0
1,modic,"{'0': 1058, '2': 149, '1': 24, '3': 4}",0
2,disc_herniation,"{'0': 1147, '1': 88}",0
3,disc_bulging,"{'0.0': 906, '1.0': 328, 'nan': 1}",1
4,disc_narrowing,"{'0': 1176, '1': 59}",0
5,spondylolisthesis,"{'0': 1194, '1': 41}",0
6,up_endplate,"{'0': 1120, '1': 115}",0
7,low_endplate,"{'0': 1163, '1': 72}",0


In [3]:
def nonempty(series):
    return series.astype(str).str.strip().replace({'nan': ''}).ne('').any()
report_counts = {field: int(master.groupby('patient_id')[field].apply(nonempty).sum()) for field in ['report_vi_findings','report_vi_impression','report_en']}
folds = {f'fold{i}': master.groupby('patient_id')[f'fold{i}_split'].first().value_counts().to_dict() for i in range(1, 6)}
print({'reports': report_counts, 'folds': folds, 'duplicate_keys': int(master.duplicated(['patient_id','level']).sum())})

{'reports': {'report_vi_findings': 238, 'report_vi_impression': 230, 'report_en': 236}, 'folds': {'fold1': {'train': 147, 'test': 50, 'val': 50}, 'fold2': {'train': 147, 'test': 50, 'val': 50}, 'fold3': {'train': 148, 'val': 50, 'test': 49}, 'fold4': {'train': 148, 'val': 50, 'test': 49}, 'fold5': {'train': 148, 'val': 50, 'test': 49}}, 'duplicate_keys': 0}


In [4]:
audit = ROOT / 'output/annotation_audit/summary.json'
if audit.exists():
    summary = json.loads(audit.read_text(encoding='utf-8'))
    print({'patient_any_conflict': summary.get('patient_any_conflict'), 'priority_A': summary.get('review_queue', {}).get('A_patient_or_internal'), 'priority_B': summary.get('review_queue', {}).get('B_level_or_definition')})
else:
    print('Audit summary not found; run scripts/audit_annotation_consistency.py first.')

{'patient_any_conflict': 131, 'priority_A': 43, 'priority_B': 112}


## Interpretation

Missing grading remains unknown. The audit conflict counts are review priorities, not automatic label errors. The complete narrative interpretation is in `docs/data_overview_report_2026-09-18.md`.